[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-orange?logo=googlecolab)](https://colab.research.google.com/github/YOUR_USERNAME/satellite-change-detection/blob/main/notebooks/07_dask_inference.ipynb)

# 07 Dask Inference

Run full-resolution sliding-window inference with Dask-backed tile parallelism.

**Prerequisites:** Google Colab GPU runtime, LEVIR-CD dataset access, repo files available.

**Expected runtime:** 10-20 minutes depending on the number of test images


## Sliding-Window Inference

This notebook runs the trained model on full 1024x1024 image pairs using weighted overlapping tiles so we do not lose edge information during inference.

In [ ]:
from pathlib import Path
import torch
from PIL import Image
from src.factory import load_config, build_model
from src.dataset import find_data_root
from src.inference import dask_sliding_window_inference
from src.utils import select_device

config = load_config(Path('configs/full.yaml'))
device = select_device('cuda')
model = build_model(config).to(device)
model.load_state_dict(torch.load('/content/siamese_cbam_best.pth', map_location=device))
layout = find_data_root(Path('/content/levir_data'))
test_a = sorted((layout.root / 'test' / layout.folder_a).glob('*.png'))[:5]
test_b = sorted((layout.root / 'test' / layout.folder_b).glob('*.png'))[:5]
test_m = sorted((layout.root / 'test' / layout.folder_label).glob('*.png'))[:5]
results = []
for path_a, path_b, path_m in zip(test_a, test_b, test_m):
    prob, pred = dask_sliding_window_inference(model, Image.open(path_a), Image.open(path_b), device, tile_size=256, overlap=128, threshold=0.5)
    results.append((path_a, path_b, path_m, prob, pred))
print('Processed', len(results), 'full-resolution pairs')
# Expected output: five full-resolution predictions computed with Dask-aware tiling.


## Full-Resolution Visualisation And Comparison

This cell renders the requested five-column grid and gives you a place to compare cropped evaluation against full-resolution inference quality.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(len(results), 5, figsize=(20, 4 * len(results)))
for row, (path_a, path_b, path_m, prob, pred) in enumerate(results):
    axes[row,0].imshow(Image.open(path_a))
    axes[row,1].imshow(Image.open(path_b))
    axes[row,2].imshow(Image.open(path_m), cmap='gray')
    axes[row,3].imshow(prob, cmap='viridis', vmin=0, vmax=1)
    axes[row,4].imshow(pred, cmap='gray')
    for col in range(5):
        axes[row,col].axis('off')
plt.tight_layout()
plt.show()
# Expected output: 5-row grid with Time1, Time2, GT, probability map, and binary prediction at 1024x1024 resolution.


## Notebook Summary

The model is now evaluated on full-resolution image pairs with weighted overlapping tiles, which lets you compare crop-based validation scores against realistic 1024x1024 deployment behavior.